In [1]:
"""
Rerun Konsolidasi: 7 Model dalam Satu Alur Narasi Linear
===========================================================
Model 1 (Baseline)
  -> A4 (Parsimoni: OwnPrice+PriceRatio+IsWeekend)
  -> A2 (+Friday +interaksi, tidak signifikan)
  -> A3 (+IsHoliday sempit, overfit)
  -> A5 (+IsHighSeason lebar, OwnPrice goyah)
  -> B1 (kembali ke level: OwnPrice+CompetitorPrice+IsHighSeason)
  -> C1 (log-log, FINAL)

Setiap model mencetak: diagnostic report lengkap, coefficient table OLS
klasik, coefficient table HAC (biar signifikansi OwnPrice/turunannya
bisa dilacak di SETIAP tahap, bukan cuma di titik yang "kebetulan" dicek),
serta train_fit_report + validation_report + train_test_comparison
(raw & clipped sekaligus).
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm

from ols_diagnostics import (
    full_diagnostic_report, validation_report,
    train_fit_report, train_test_comparison
)

# ============================================================
# LOAD & FEATURE ENGINEERING
# ============================================================

df = pd.read_csv('pullman_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['IsFriday'] = (df['DayOfWeek'] == 4).astype(int)

df['PriceRatio'] = df['OwnPrice (Rp)'] / df['CompetitorPrice (Rp)'].replace(0, np.nan)
df['OwnPrice_centered'] = df['OwnPrice (Rp)'] - df['OwnPrice (Rp)'].mean()
df['Price_Weekend'] = df['OwnPrice_centered'] * df['IsWeekend']

df['ln_OwnPrice'] = np.log(df['OwnPrice (Rp)'].replace(0, np.nan))
df['ln_CompetitorPrice'] = np.log(df['CompetitorPrice (Rp)'].replace(0, np.nan))

# IsHoliday window SEMPIT (A3) - hanya gap cuti biasa, TERMASUK Jun-Jul
df['IsHoliday_narrow'] = (
    (df['Date'].dt.month.isin([6, 7])) |
    ((df['Date'].dt.month == 12) & (df['Date'].dt.day >= 20)) |
    ((df['Date'].dt.month == 1) & (df['Date'].dt.day <= 5))
).astype(int)

# IsHighSeason window LEBAR (A5, B1, C1) - LOCKED, dari training pattern + hint soal
m, d = df['Date'].dt.month, df['Date'].dt.day
df['IsHighSeason'] = (
    ((m == 1) & (d <= 7)) | (m == 6) | (m == 7) | ((m == 12) & (d >= 16))
).astype(int)

SPLIT_DATE = pd.Timestamp('2025-10-01')

def split(cols):
    X = df[cols]
    y = df['Occupancy (%)']
    Xtr = X[df['Date'] < SPLIT_DATE]
    Xte = X[df['Date'] >= SPLIT_DATE]
    ytr = y[df['Date'] < SPLIT_DATE]
    yte = y[df['Date'] >= SPLIT_DATE]
    return Xtr, Xte, ytr, yte


def run_model(cols, label, hac_maxlags=7):
    """Jalankan satu model lengkap: diagnostic + OLS coef + HAC coef + train/test comparison."""
    Xtr, Xte, ytr, yte = split(cols)

    model, _ = full_diagnostic_report(Xtr, ytr, feature_names=cols, model_label=label)

    # Coefficient table OLS klasik (ringkas, khusus buat lacak signifikansi per tahap)
    print("\n" + "-" * 70)
    print(f"RINGKASAN COEF OLS KLASIK - {label}")
    print("-" * 70)
    ols_table = pd.DataFrame({
        "Coef": model.params, "Std Error": model.bse,
        "t/z": model.tvalues, "p-value": model.pvalues,
    })
    print(ols_table.round(4).to_string())

    # HAC
    Xtr_const = sm.add_constant(Xtr)
    model_hac = sm.OLS(ytr, Xtr_const).fit(cov_type='HAC', cov_kwds={'maxlags': hac_maxlags})
    print("\n" + "-" * 70)
    print(f"RINGKASAN COEF HAC - {label}")
    print("-" * 70)
    hac_table = pd.DataFrame({
        "Coef": model_hac.params, "Std Error": model_hac.bse,
        "z": model_hac.tvalues, "p-value": model_hac.pvalues,
    })
    print(hac_table.round(4).to_string())
    print()

    train_result = train_fit_report(model, Xtr, ytr, label=f"{label} - Training")
    test_result = validation_report(model, Xte, yte, label=f"{label} - Okt-Des 2025")
    train_test_comparison(train_result, test_result)

    return model, model_hac, train_result, test_result


results_all = {}

# ============================================================
# MODEL 1: BASELINE
# ============================================================
results_all['Model 1 (Baseline)'] = run_model(
    ['OwnPrice (Rp)', 'CompetitorPrice (Rp)', 'Month'],
    "Model 1 - Baseline"
)

# ============================================================
# A4: PARSIMONI (OwnPrice + PriceRatio + IsWeekend)
# ============================================================
# (VIF check OwnPrice+CompPrice+PriceRatio bareng - bukti kenapa CompPrice di-drop)
from statsmodels.stats.outliers_influence import variance_inflation_factor
print("\n\n" + "=" * 70)
print("CEK VIF: OwnPrice + CompetitorPrice + PriceRatio (sebelum drop CompPrice)")
print("=" * 70)
Xtr_check, _, _, _ = split(['OwnPrice (Rp)', 'CompetitorPrice (Rp)', 'PriceRatio'])
X_check_const = sm.add_constant(Xtr_check)
for i, col in enumerate(X_check_const.columns):
    print(f"  {col}: VIF={variance_inflation_factor(X_check_const.values, i):.3f}")
print()

results_all['A4 (Parsimoni)'] = run_model(
    ['OwnPrice (Rp)', 'PriceRatio', 'IsWeekend'],
    "A4 - Parsimoni"
)

# ============================================================
# A2: + FRIDAY + INTERAKSI (dicoba, tidak signifikan)
# ============================================================
results_all['A2 (+Friday+Interaksi)'] = run_model(
    ['OwnPrice (Rp)', 'PriceRatio', 'IsFriday', 'IsWeekend', 'Price_Weekend'],
    "A2 - Friday + Interaksi"
)

# ============================================================
# A3: + ISHOLIDAY SEMPIT (overfit)
# ============================================================
results_all['A3 (+IsHoliday sempit)'] = run_model(
    ['OwnPrice (Rp)', 'PriceRatio', 'IsWeekend', 'IsHoliday_narrow'],
    "A3 - IsHoliday Sempit"
)

# ============================================================
# A5: + ISHIGHSEASON LEBAR (OwnPrice goyah)
# ============================================================
results_all['A5 (+IsHighSeason lebar)'] = run_model(
    ['OwnPrice (Rp)', 'PriceRatio', 'IsWeekend', 'IsHighSeason'],
    "A5 - IsHighSeason Lebar"
)

# ============================================================
# B1: KEMBALI KE LEVEL (tanpa turunan rasio)
# ============================================================
results_all['B1 (Level)'] = run_model(
    ['OwnPrice (Rp)', 'CompetitorPrice (Rp)', 'IsWeekend', 'IsHighSeason'],
    "B1 - Level"
)

# ============================================================
# C1: LOG-LOG (FINAL)
# ============================================================
results_all['C1 (Log-Log) - FINAL'] = run_model(
    ['ln_OwnPrice', 'ln_CompetitorPrice', 'IsWeekend', 'IsHighSeason'],
    "C1 - Log-Log (FINAL)"
)


# ============================================================
# RINGKASAN AKHIR: JEJAK SIGNIFIKANSI OwnPrice DI SETIAP TAHAP
# ============================================================
print("\n\n" + "#" * 70)
print("#  JEJAK SIGNIFIKANSI OwnPrice (atau turunannya) DI SETIAP TAHAP")
print("#" * 70)
print(f"{'Model':30s}{'p OLS':>12s}{'p HAC':>12s}{'Adj R2 Tr':>12s}{'R2 Val Raw':>12s}{'AIC':>10s}")
own_price_names = {
    'Model 1 (Baseline)': 'OwnPrice (Rp)',
    'A4 (Parsimoni)': 'OwnPrice (Rp)',
    'A2 (+Friday+Interaksi)': 'OwnPrice (Rp)',
    'A3 (+IsHoliday sempit)': 'OwnPrice (Rp)',
    'A5 (+IsHighSeason lebar)': 'OwnPrice (Rp)',
    'B1 (Level)': 'OwnPrice (Rp)',
    'C1 (Log-Log) - FINAL': 'ln_OwnPrice',
}
for name, (model, model_hac, train_r, test_r) in results_all.items():
    var = own_price_names[name]
    p_ols = model.pvalues[var]
    p_hac = model_hac.pvalues[var]
    print(f"{name:30s}{p_ols:12.4f}{p_hac:12.4f}{model.rsquared_adj:12.4f}{test_r['raw']['r2']:12.4f}{model.aic:10.2f}")


######################################################################
#  FULL DIAGNOSTIC REPORT - Model 1 - Baseline
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                      OwnPrice (Rp)  CompetitorPrice (Rp)  Month
OwnPrice (Rp)                 1.000                 0.429  0.042
CompetitorPrice (Rp)          0.429                 1.000  0.115
Month                         0.042                 0.115  1.000

--- Korelasi tiap X terhadap Y (Occupancy) ---
            Variabel      r  p-value Sig
       OwnPrice (Rp) 0.2582   0.0000 ***
CompetitorPrice (Rp) 0.4746   0.0000 ***
               Month 0.0335   0.5812    

2. MODEL SUMMARY (Coefficients, t-test, F-test)
                        Coef  Std Error  t-stat  p-value  Sig
const                -0.9419     8.3769 -0.1124   0.9106     
OwnPrice (Rp)         0.0000     0.0000  1.1274   0.2606     
CompetitorPrice (Rp)  0.0000     0.0000  7.5

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)
c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusc

Threshold Cook's Distance : 0.0147  (rule of thumb: 4/n)
Threshold Leverage        : 0.0440  (rule of thumb: 2k/n)
Threshold Studentized Res : |t| > 3

Total observasi terflag (minimal satu kriteria): 27 dari 273

Top 10 observasi paling mencurigakan (urut by Cook's D):
     Cooks_D  Leverage  Student_Resid  Flag_Cooks  Flag_Leverage  Flag_Outlier
87    0.0900    0.1143         2.0586        True           True         False
192   0.0304    0.0844         1.4098        True           True         False
240   0.0187    0.0985         1.0139        True           True         False
184   0.0186    0.0273         2.0032        True          False         False
198   0.0174    0.0302         1.8380        True          False         False
226   0.0172    0.0364         1.6614        True          False         False
142   0.0156    0.0262        -1.8754        True          False         False
180   0.0154    0.0182         2.2490        True          False         False
113   0.0147    0.

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)
c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusc

AIC               : 2397.13
BIC               : 2415.18
N observations    : 273

3. UJI NORMALITAS RESIDUAL
Shapiro-Wilk       : W=0.9875, p-value=0.0178  -> TIDAK normal (tolak H0)
Kolmogorov-Smirnov : D=0.0546, p-value=0.3761  -> Normal (gagal tolak H0)
Jarque-Bera        : JB=4.3464, p-value=0.1138  -> Normal (gagal tolak H0)
  (Skewness=0.2193, Kurtosis=2.5645)

H0: residual berdistribusi normal | Ditolak jika p-value < 0.05

4. UJI HOMOSKEDASTISITAS
Breusch-Pagan  : LM stat=42.4858, p-value=0.0000  -> Heteroskedastis (tolak H0)
Glejser        : F-stat=10.9243, p-value(F)=0.0000  -> Heteroskedastis (tolak H0)

  Detail Glejser per variabel (p-value < 0.05 = variabel itu sumber heteroskedastisitas):
                        Coef  p-value  Sig
const                 2.8480   0.4352     
OwnPrice (Rp)         0.0000   0.0688    .
CompetitorPrice (Rp)  0.0000   0.0035   **
IsWeekend            -8.9575   0.0000  ***
IsHighSeason         -0.1883   0.9050     

H0: varians residual konstan 

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)
